# 🎬 AutoTube AI - 1-Click Mobile Cloud Studio
### 100% Free • No Laptop Needed • 1-Tap Execution

This notebook runs **AutoTube AI** on Google Cloud servers.

👉 **Instructions:**
1. Tap the **▶️ Play** button on the cell below (or tap **Runtime** -> **Run all**).
2. Wait 1-2 minutes until it prints your live mobile link.
3. Tap the link to open your studio! Keep this tab in the background.

---

In [ ]:
# ============================================================
# 1. AUTOMATIC SETUP & DEPENDENCY INSTALLATION
# ============================================================
import os, sys, time, subprocess

print('⏳ [1/3] Installing system tools (FFmpeg & audio libraries)...')
!apt-get update -qq && apt-get install -y -qq ffmpeg libsndfile1 > /dev/null

print('⏳ [2/3] Downloading AutoTube AI project files...')
!wget -q https://huggingface.co/datasets/harsha7981/autotube-colab/resolve/main/autotube_colab.zip -O /content/autotube_colab.zip
!rm -rf /content/autotube-ai && unzip -q -o /content/autotube_colab.zip -d /content/autotube-ai

print('⏳ [3/3] Installing Python packages from requirements.txt...')
!pip install -q -r /content/autotube-ai/requirements.txt pyngrok

print('✅ Setup complete!')

# ============================================================
# 2. LAUNCH STREAMLIT & NGROK MOBILE TUNNEL
# ============================================================
from dotenv import load_dotenv
from pyngrok import ngrok

os.chdir('/content/autotube-ai')
load_dotenv('.env')

ngrok_token = os.getenv('NGROK_AUTHTOKEN')
ngrok_domain = os.getenv('NGROK_STATIC_DOMAIN')
app_pin = os.getenv('APP_PIN', '9989')

if not ngrok_token:
    raise ValueError('NGROK_AUTHTOKEN not found in .env!')

# Clean up any stale processes
ngrok.set_auth_token(ngrok_token)
ngrok.kill()
!pkill -f streamlit || true
time.sleep(1)

# Start Streamlit
print('🚀 Starting Streamlit server...')
cmd = 'streamlit run app.py --server.port 8501 --server.address 0.0.0.0 --server.headless true --server.enableCORS false --server.enableXsrfProtection false'
streamlit_proc = subprocess.Popen(cmd, shell=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(5)

# Connect Ngrok tunnel
print('🌐 Connecting Ngrok tunnel...')
tunnel = ngrok.connect(8501, 'http', domain=ngrok_domain)

print('\n' + '=' * 60)
print('🎉 AutoTube AI is now LIVE on Google Cloud!')
print(f'📱 Mobile Link   : {tunnel.public_url}')
print(f'🔑 Security PIN  : {app_pin}')
print('=' * 60 + '\n')
print('👉 Keep this tab open in the background while you use your mobile link!')
print('⏳ Server is running...')

try:
    while True:
        time.sleep(60)
except KeyboardInterrupt:
    print('Stopping server...')
    ngrok.kill()
    streamlit_proc.terminate()
